In [ ]:
import numpy as np
import pandas as pd
from typing import Optional


def preprocess_sleep_data(input_path: str = "input.csv",
                          save_path: Optional[str] = None) -> pd.DataFrame:
    """
    Load and preprocess sleep disorder dataset.

    Steps:
    1. Load CSV
    2. Keep only the specified 10 features + target
    3. Map categorical variables:
       - Gender: Male -> 1, Female -> 0, else NaN
       - Snoring: TRUE/YES/1 -> 1, FALSE/NO/0 -> 0, else NaN
       - Diagnosis_of_SDB (None/Mild/Moderate/Severe):
            None      -> 0 (no SDB)
            Mild+     -> 1 (SDB present)
    4. Report missing values per column
    5. Optionally save the processed DataFrame as CSV
    """

    # 1. Load CSV
    df = pd.read_csv(input_path)

    # 2. Select only the required columns
    expected_columns = [
        "Age",
        "Gender",
        "BMI",
        "Snoring",
        "Oxygen_Saturation",
        "AHI",
        "ECG_Heart_Rate",
        "SpO2",
        "Nasal_Airflow",
        "Chest_Movement",
        "Diagnosis_of_SDB",
    ]

    missing_cols = [c for c in expected_columns if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"The following required columns are missing in the input file: {missing_cols}"
        )

    df = df[expected_columns].copy()

    # ---------- 3. Preprocess categorical variables ----------

    # 3.1 Gender: Male -> 1, Female -> 0, unexpected/missing -> NaN
    def map_gender(val):
        if pd.isna(val):
            return np.nan
        s = str(val).strip().lower()
        if s == "male":
            return 1
        if s == "female":
            return 0
        return np.nan  # unexpected value

    df["Gender"] = df["Gender"].apply(map_gender)

    # 3.2 Snoring: TRUE/YES/1 -> 1, FALSE/NO/0 -> 0, unexpected/missing -> NaN
    def map_snoring(val):
        if pd.isna(val):
            return np.nan
        # handle real booleans
        if isinstance(val, bool):
            return 1 if val else 0
        s = str(val).strip().lower()
        if s in ("true", "yes", "1"):
            return 1
        if s in ("false", "no", "0"):
            return 0
        return np.nan  # unexpected value

    df["Snoring"] = df["Snoring"].apply(map_snoring)

    # 3.3 Diagnosis_of_SDB:
    #     None      -> 0 (no SDB)
    #     Mild/Moderate/Severe -> 1 (SDB present)
    def map_sdb_presence(val):
        if pd.isna(val):
            return np.nan
        s = str(val).strip().lower()
        if s == "none":
            return 0
        if s in ("mild", "moderate", "severe", "yes", "present"):
            return 1
        return np.nan  # unexpected label

    df["Diagnosis_of_SDB"] = df["Diagnosis_of_SDB"].apply(map_sdb_presence)

    # ---------- 4. Report missing values ----------
    print("\nMissing values per column after preprocessing:")
    missing_counts = df.isna().sum()
    print(missing_counts)

    # ---------- 5. Optionally save to CSV ----------
    if save_path is not None:
        df.to_csv(save_path, index=False)
        print(f"\nPreprocessed dataset saved to: {save_path}")

    # ---------- 6. Preview ----------
    print("\nPreview of the preprocessed DataFrame:")
    print(df.head())

    return df


if __name__ == "__main__":
    # Example usage
    processed_df = preprocess_sleep_data(
        input_path="sdb_dataset.csv",        # or "input.csv"
        save_path="preprocessed_sdb.csv"     # or None if you don't want to save
    )
